CELL1: install libraries


In [2]:
# Comment 1: Install MNE for reading EEG recordings and NumPy for numerical data.
# Comment 2: Install WFDB to access PhysioNet datasets and Matplotlib for plotting.

!pip -q install mne wfdb numpy matplotlib pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 17.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


CELL2: Import Libraries

In [6]:
# Comment 1: Import the libraries required for downloading and processing EEG data.
# Comment 2: These libraries will be used in the following cells.

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
import wfdb
import tensorflow as tf

print("Libraries imported successfully")

# Comment 1: Set a fixed random seed so the experiment can be reproduced.
# Comment 2: Reproducibility helps us compare different model versions.

np.random.seed(42)
tf.random.set_seed(42)

print("Random seed set")

Libraries imported successfully
Random seed set


CELL3: Set Random Seed

CELL4: Enter dataset path

In [ ]:
# Comment 1: Create a folder inside Colab to store the downloaded EEG dataset.
# Comment 2: The folder will be used as the dataset path in later cells.

DATASET_PATH = "/content/erpbci"

os.makedirs(DATASET_PATH, exist_ok=True)

print("Dataset folder created:")
print(DATASET_PATH)

# Comment 1: Download the PhysioNet ERPBci dataset into the Colab folder.
# Comment 2: The -r option downloads the dataset folders and their files.

!wget -r -N -c -np \
  https://physionet.org/files/erpbci/1.0.0/ \
  -P /content/erpbci

Dataset folder created:
/content/erpbci
--2026-09-15 15:59:11--  https://physionet.org/files/erpbci/1.0.0/
Resolving physionet.org (physionet.org)... 18.25.8.254
Connecting to physionet.org (physionet.org)|18.25.8.254|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘/content/erpbci/physionet.org/files/erpbci/1.0.0/index.html’

physionet.org/files     [ <=>                ]   2.05K  --.-KB/s    in 0s      

Last-modified header missing -- time-stamps turned off.
2026-09-15 15:59:12 (391 MB/s) - ‘/content/erpbci/physionet.org/files/erpbci/1.0.0/index.html’ saved [2100]

Loading robots.txt; please ignore errors.
--2026-09-15 15:59:12--  https://physionet.org/robots.txt
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 22 [text/plain]
Saving to: ‘/content/erpbci/physionet.org/robots.txt’

physionet.org/robot 100%[===================>]      22  --.-KB/s    in 0s      

2026-09

CELL5 : Check downloaded files

In [ ]:
# Comment 1: Search for EDF files in the downloaded PhysioNet dataset.
# Comment 2: Print the number of recordings and a few file paths to verify download.

edf_files = glob.glob(
    "/content/erpbci/**/*.edf",
    recursive=True
)

print("Number of EDF files:", len(edf_files))

for file in edf_files[:10]:
    print(file)

CELL 6: Select one EDF filrecording

In [ ]:
# Comment 1: Select the first EDF recording for initial EEG inspection.
# Comment 2: We will use this file to verify that MNE can read the dataset.

assert len(edf_files) > 0, "No EDF files found. Check the download."

EDF_FILE = edf_files[0]

print("Selected file:")
print(EDF_FILE)

CELL8: Load the EEG file

In [ ]:
# Comment 1: Read the EDF+ EEG recording using MNE.
# Comment 2: preload=True loads the signal into memory for preprocessing.

raw = mne.io.read_raw_edf(
    EDF_FILE,
    preload=True,
    verbose=False
)

print(raw)
print("Sampling frequency:", raw.info["sfreq"])
print("Number of channels:", len(raw.ch_names))
print("Recording duration:", raw.times[-1], "seconds")

CELL9: Display channel names

In [ ]:
# Comment 1: Print all channel names so we can identify EEG and EOG channels.
# Comment 2: The channel names help us select the correct signals for CNN.

print("Channel names:")

for index, name in enumerate(raw.ch_names):
    print(index, name)

CELL: Select EEG channels

In [ ]:
# Comment 1: Select EEG channels and exclude EOG and reference channels.
# Comment 2: We use 8 EEG channels initially to reduce computation time.

eeg_channels = [
    ch for ch in raw.ch_names
    if ch.upper().startswith("A")
]

print("Selected EEG channels:", eeg_channels)
print("Number of selected channels:", len(eeg_channels))

Cell10: Keep only EEG channels

In [ ]:
# Comment 1: Pick the EEG channels that will be used for preprocessing.
# Comment 2: The resulting raw object contains only the selected EEG signals.

raw_eeg = raw.copy().pick(eeg_channels)

print(raw_eeg)
print("EEG channels:", raw_eeg.ch_names)

CELL11: Plot original EEG signal

In [ ]:
# Comment 1: Extract a short segment of the original EEG signal for plotting.
# Comment 2: The plot helps us see the signal before preprocessing.

data_original, times_original = raw_eeg.get_data(
    start=0,
    stop=min(int(raw_eeg.info["sfreq"] * 5), raw_eeg.n_times),
    return_times=True
)

plt.figure(figsize=(14, 6))

plt.plot(
    times_original,
    data_original[0] * 1e6
)

plt.title("Original EEG Signal - First Channel")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude (µV)")
plt.grid(True)
plt.show()

CELL12: Re-reference EEG

In [ ]:
# Comment 1: Apply average referencing to reduce common signal components.
# Comment 2: Average referencing improves the reference used for EEG analysis.

raw_eeg.set_eeg_reference("average", projection=False)

print("Average reference applied")

CELL13: Downsample EEG

In [ ]:
# Comment 1: Reduce the sampling frequency to 256 Hz to make training faster.
# Comment 2: The reduced rate is sufficient for this prototype's ERP time window.

raw_eeg.resample(256)

print("New sampling frequency:", raw_eeg.info["sfreq"])

CELL14: Apply band-pass filter

In [ ]:
# Comment 1: Keep EEG frequencies between 0.1 and 20 Hz for the ERP prototype.
# Comment 2: Filtering reduces unwanted high-frequency signal components.

raw_eeg.filter(
    l_freq=0.1,
    h_freq=20.0,
    verbose=False
)

print("Band-pass filter applied: 0.1–20 Hz")

CELL15: Plot filtered EEG

In [ ]:
# Comment 1: Extract the first five seconds of the filtered EEG signal.
# Comment 2: Compare this waveform with the original signal from Cell 11.

data_filtered, times_filtered = raw_eeg.get_data(
    start=0,
    stop=min(int(raw_eeg.info["sfreq"] * 5), raw_eeg.n_times),
    return_times=True
)

plt.figure(figsize=(14, 6))

plt.plot(
    times_filtered,
    data_filtered[0] * 1e6
)

plt.title("Filtered EEG Signal - First Channel")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude (µV)")
plt.grid(True)
plt.show()

CELL16: Read annotations

In [ ]:
# Comment 1: Extract annotation descriptions and their event timings.
# Comment 2: These descriptions tell us which character groups were flashed.

annotations = raw_eeg.annotations

annotation_descriptions = list(annotations.description)
annotation_onsets = list(annotations.onset)

print("Number of annotations:", len(annotation_descriptions))

for onset, desc in zip(
    annotation_onsets[:20],
    annotation_descriptions[:20]
):
    print(round(onset, 3), desc)

CELL17:Extract target character

In [ ]:
# Comment 1: Find the target character from the target annotation.
# Comment 2: The target character is used to label target and non-target flashes.

target_character = None

for desc in annotation_descriptions:

    match = re.search(
        r"#Tgt([A-Za-z0-9])_",
        desc
    )

    if match:
        target_character = match.group(1).upper()
        break

print("Target character:", target_character)

assert target_character is not None, (
    "Target character not found in annotations."
)

CELL18: Display all flash events

In [ ]:
# Comment 1: Display the annotations that describe flashed character groups.
# Comment 2: These events will be used to create the EEG training epochs.

flash_events = []

for onset, desc in zip(
    annotation_onsets,
    annotation_descriptions
):

    if not desc.startswith("#"):

        flash_events.append({
            "onset": onset,
            "description": desc
        })

print("Number of flash events:", len(flash_events))

for event in flash_events[:20]:
    print(event)

CELL19: Create ON/OFF labels

In [ ]:
# Comment 1: Label a flash ON when it contains the target character.
# Comment 2: Label a flash OFF when it does not contain the target character.

for event in flash_events:

    event["label"] = int(
        target_character in event["description"].upper()
    )

print("First five labeled events:")

for event in flash_events[:5]:
    print(event)

CELL20: Check class balance

In [ ]:
# Comment 1: Count the number of ON and OFF labels in the selected recording.
# Comment 2: Class counts help us understand whether the dataset is balanced.

labels_all = np.array([
    event["label"]
    for event in flash_events
])

print("OFF samples:", np.sum(labels_all == 0))
print("ON samples:", np.sum(labels_all == 1))
print("Total samples:", len(labels_all))

CELL21: Define epoch settings

In [ ]:
# Comment 1: Set the EEG time window used for each flash event.
# Comment 2: Each epoch will contain EEG after the stimulus onset.

SFREQ = raw_eeg.info["sfreq"]

TMIN = 0.0
TMAX = 1.0

N_SAMPLES = int((TMAX - TMIN) * SFREQ)

print("Sampling frequency:", SFREQ)
print("Samples per epoch:", N_SAMPLES)

CELL22: Extract EEG epochs

In [ ]:
# Comment 1: Extract one second of EEG following every flash event.
# Comment 2: Each extracted epoch will become one CNN training sample.

X_list = []
y_list = []

for event in flash_events:

    onset = event["onset"]

    start_sample = int(onset * SFREQ)
    stop_sample = start_sample + N_SAMPLES

    if stop_sample <= raw_eeg.n_times:

        epoch_data = raw_eeg.get_data(
            start=start_sample,
            stop=stop_sample
        )

        X_list.append(epoch_data)
        y_list.append(event["label"])

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int32)

print("X shape:", X.shape)
print("y shape:", y.shape)

CELL23: Check the epoch shape

In [ ]:
# Comment 1: Print the number of samples, channels, and time points.
# Comment 2: Verify that the CNN input contains the expected EEG dimensions.

print("Number of epochs:", X.shape[0])
print("Number of channels:", X.shape[1])
print("Number of time points:", X.shape[2])

CELL24: Plot onre EEG epoch

In [ ]:
# Comment 1: Plot the first channel of the first EEG epoch.
# Comment 2: This visualizes the signal that will be given to the CNN.

plt.figure(figsize=(12, 5))

plt.plot(
    X[0, 0, :] * 1e6
)

plt.title("One EEG Epoch")
plt.xlabel("Time samples")
plt.ylabel("Amplitude (µV)")
plt.grid(True)
plt.show()

Cell 25 — Normalize EEG values

In [ ]:
# Comment 1: Normalize each EEG epoch using its mean and standard deviation.
# Comment 2: Normalization helps the CNN train more consistently.

mean_values = X.mean(axis=2, keepdims=True)
std_values = X.std(axis=2, keepdims=True)

X = (X - mean_values) / (std_values + 1e-8)

print("Normalization complete")
print("Mean:", X.mean())
print("Standard deviation:", X.std())

Cell 26 — Convert EEG into CNN format

In [ ]:
# Comment 1: Move the time dimension before the channel dimension.
# Comment 2: TensorFlow Conv1D expects input in samples, time, channels format.

X = np.transpose(X, (0, 2, 1))

print("New CNN input shape:", X.shape)

Cell 27 — Train/test split

In [ ]:
# Comment 1: Split the EEG samples into training and testing datasets.
# Comment 2: Stratification preserves the ON/OFF class ratio in both sets.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Cell 28 — Create CNN architecture

In [ ]:
# Comment 1: Build a 1D CNN that learns patterns from EEG time sequences.
# Comment 2: The final sigmoid neuron predicts the probability of ON.

model = models.Sequential([

    layers.Input(
        shape=(X_train.shape[1], X_train.shape[2])
    ),

    layers.Conv1D(
        filters=32,
        kernel_size=7,
        activation="relu",
        padding="same"
    ),

    layers.BatchNormalization(),

    layers.MaxPooling1D(
        pool_size=2
    ),

    layers.Conv1D(
        filters=64,
        kernel_size=5,
        activation="relu",
        padding="same"
    ),

    layers.BatchNormalization(),

    layers.MaxPooling1D(
        pool_size=2
    ),

    layers.GlobalAveragePooling1D(),

    layers.Dense(
        32,
        activation="relu"
    ),

    layers.Dropout(0.3),

    layers.Dense(
        1,
        activation="sigmoid"
    )
])

model.summary()

Cell 29 — Compile the model

In [ ]:
# Comment 1: Configure binary cross-entropy loss for two-class prediction.
# Comment 2: Adam updates CNN weights during training.

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("CNN compiled successfully")

Cell 30 — Train the CNN

In [ ]:
# Comment 1: Train the CNN using the EEG training samples and ON/OFF labels.
# Comment 2: Validation data monitors performance on unseen training samples.

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Cell 31 — Plot training accuracy

In [ ]:
# Comment 1: Plot training and validation accuracy during CNN learning.
# Comment 2: The graph helps us see whether the model is learning.

plt.figure(figsize=(10, 5))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.title("CNN Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

Cell 32 — Evaluate the CNN

In [ ]:
# Comment 1: Evaluate the trained CNN using the unseen testing dataset.
# Comment 2: The test accuracy measures how well the model predicts labels.

test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

Cell 33 — Generate predictions

In [ ]:
# Comment 1: Predict ON probability for every test EEG epoch.
# Comment 2: Convert the probabilities into binary ON/OFF predictions.

probabilities = model.predict(
    X_test,
    verbose=0
).ravel()

y_pred = (probabilities >= 0.5).astype(int)

print("Predictions generated")
print("First probabilities:", probabilities[:10])
print("First predictions:", y_pred[:10])

Cell 34 — Print classification report

In [ ]:
# Comment 1: Display precision, recall, and F1-score for ON and OFF classes.
# Comment 2: These metrics provide more information than accuracy alone.

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["OFF", "ON"],
        zero_division=0
    )
)

Cell 35 — Plot confusion matrix

In [ ]:
# Comment 1: Calculate the confusion matrix for ON and OFF predictions.
# Comment 2: The matrix shows correct and incorrect predictions per class.

cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(6, 5))

plt.imshow(cm)

plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.xticks(
    [0, 1],
    ["OFF", "ON"]
)

plt.yticks(
    [0, 1],
    ["OFF", "ON"]
)

for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.show()

Cell 36 — Predict one EEG sample

In [ ]:
# Comment 1: Select one EEG epoch from the testing dataset for prediction.
# Comment 2: The CNN returns the probability that the appliance should be ON.

sample_index = 0

sample = X_test[sample_index:sample_index + 1]

prediction_probability = model.predict(
    sample,
    verbose=0
)[0][0]

print("ON probability:", prediction_probability)

Cell 37 — Display the final ON/OFF result

In [ ]:
# Comment 1: Convert the CNN probability into an ON or OFF appliance decision.
# Comment 2: Display the final predicted appliance state for the prototype.

threshold = 0.5

if prediction_probability >= threshold:

    appliance_state = "APPLIANCE ON"

else:

    appliance_state = "APPLIANCE OFF"

print("================================")
print("     NEUROHOME PROTOTYPE")
print("================================")
print("Predicted probability:", prediction_probability)
print("Final output:", appliance_state)
print("================================")

Cell 38 — Save CNN

In [ ]:
# Comment 1: Save the trained CNN model to a file for later use.
# Comment 2: The saved model can be loaded in a future NeuroHome application.

model.save(
    "/content/neurohome_cnn.keras"
)

print("Model saved successfully")

Cell 39 — Download the model

In [ ]:
# Comment 1: Import the Colab download utility for saving the trained model.
# Comment 2: Download the model file to your computer for project use.

from google.colab import files

files.download(
    "/content/neurohome_cnn.keras"
)